# Model 2 · EfficientNet-B0 for DataSEC
**Venisa Ivan Tellis · 230911064 · ICT 4442**

This notebook runs only the EfficientNet-B0 modules. It extends the existing Model 2 implementation. Full DataSEC training and verified project scores are pending; saved outputs in this notebook are intentionally empty.

**Workflow:** choose a GPU runtime → set paths → reuse the team split → short check → train → choose on validation → evaluate the final checkpoint → inspect errors and export.

ChatGPT Codex assisted with this notebook and the accompanying code changes. Review the implementation, conduct the experiments and disclose this assistance in your report.


## 1. Open the Model 2 code and install dependencies
In Kaggle, enable Internet for installation and optional dataset download, and select a GPU for training. In Google Colab, select a GPU runtime. In a local checkout, open this notebook from `EfficientNet-B0/`.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/enigmatronix13/Sound-Event-Classification.git"
BRANCH = "main"  # the Model 2 implementation has been merged
current = Path.cwd()
if (current / "src/train_efficientnet.py").is_file():
    MODEL_DIR = current
elif (current / "EfficientNet-B0/src/train_efficientnet.py").is_file():
    MODEL_DIR = current / "EfficientNet-B0"
else:
    checkout = current / "Sound-Event-Classification"
    if not checkout.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(checkout)], check=True)
    MODEL_DIR = checkout / "EfficientNet-B0"
os.chdir(MODEL_DIR)
assert Path("src/evaluate_efficientnet.py").is_file(), "Open the updated Model 2 contribution branch."
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
print("Working in", Path.cwd())


## 2. Find or download the DataSEC audio
**If you have not added a dataset in Kaggle:** set `DOWNLOAD_DATASEC = True` in the next cell and run it. Enable Internet in the notebook settings first. It downloads the 6.4 GB ZIP, checks its published checksum, extracts it in writable storage and sets `DATA_ROOT`. Extraction needs additional disk space. Wait for the `DATA_ROOT = ...` line before continuing.

This optional download uses [DataSEC v3, Zenodo record 17033970](https://zenodo.org/records/17033970), matching the release reported by the team's SincNet run. If the team agrees a different release, attach that dataset instead.

**If DataSEC is already attached:** leave `DOWNLOAD_DATASEC = False`. The cell searches Kaggle inputs for the 22-class folder layout. For an unrecognized or ambiguous layout, set `DATA_ROOT` explicitly to the extracted audio directory. It does not silently choose between multiple copies.

Google Drive users can mount Drive and supply `DATA_ROOT` explicitly. The repository ZIP contains code only; it does not contain the sound recordings.


In [ ]:
# Kaggle: set True if you have not added the DataSEC audio dataset.
# This downloads DataSEC v3, the release used by the team's SincNet run.
DOWNLOAD_DATASEC = False
DATA_ROOT = None  # Or set Path('/kaggle/input/your-dataset/DATASEC') explicitly.

from pathlib import Path
import hashlib
import json
import os
import shutil
import urllib.error
import urllib.request
import zipfile
from tqdm.auto import tqdm

DATASEC_URL = 'https://zenodo.org/records/17033970/files/DATASEC.zip?download=1'
DATASEC_MD5 = '29fa9b8cc84cfa69aa4e5e674e780383'
SETUP_DIR = (Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()) / 'data' / 'datasec_v3'


def file_md5(path):
    digest = hashlib.md5()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def download_and_extract(destination, url=DATASEC_URL, expected_md5=DATASEC_MD5):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    archive = destination / 'DATASEC.zip'
    extracted = destination / 'extracted'
    marker = extracted / '.datasec_complete.json'
    if marker.is_file():
        saved = json.loads(marker.read_text())
        if saved.get('md5') == expected_md5 and any(extracted.rglob('*.wav')):
            print('Using the previously extracted dataset:', extracted)
            return extracted
    if extracted.exists():
        raise ValueError(f'Unverified extraction already exists at {extracted}. Choose a new SETUP_DIR.')
    if not archive.is_file():
        partial = archive.with_suffix('.zip.part')
        request = urllib.request.Request(url, headers={'User-Agent': 'DataSEC-EfficientNetB0/1.0'})
        try:
            with urllib.request.urlopen(request, timeout=120) as response, partial.open('wb') as output:
                size = int(response.headers.get('Content-Length', 0)) or None
                if size and shutil.disk_usage(destination).free < size:
                    raise OSError('Not enough free disk space for the DataSEC ZIP.')
                with tqdm(total=size, unit='B', unit_scale=True, desc='Downloading DataSEC') as progress:
                    while True:
                        chunk = response.read(8 * 1024 * 1024)
                        if not chunk:
                            break
                        output.write(chunk)
                        progress.update(len(chunk))
        except (urllib.error.URLError, TimeoutError) as exc:
            raise RuntimeError('Download failed. Enable Internet in Kaggle notebook settings, then rerun. '
                               'Alternatively attach the extracted DataSEC dataset as an input.') from exc
        if file_md5(partial) != expected_md5:
            raise ValueError('Downloaded ZIP checksum mismatch. Rerun to download a fresh copy.')
        partial.replace(archive)
    elif file_md5(archive) != expected_md5:
        raise ValueError(f'ZIP checksum mismatch: {archive}. Remove this invalid ZIP or use a new SETUP_DIR.')
    staging = destination / 'extracting'
    staging.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        members = z.infolist()
        required = sum(item.file_size for item in members)
        if shutil.disk_usage(destination).free < required:
            raise OSError(f'Extraction needs approximately {required / 1e9:.1f} GB of free space in {destination}.')
        for item in members:
            target = (staging / item.filename).resolve()
            if not target.is_relative_to(staging.resolve()):
                raise ValueError('Unsafe path in ZIP archive.')
        for item in tqdm(members, desc='Extracting DataSEC'):
            z.extract(item, staging)
    (staging / '.datasec_complete.json').write_text(json.dumps({'url': url, 'md5': expected_md5}))
    staging.rename(extracted)
    return extracted


def find_datasec_roots(search_roots):
    # Match known class/subclass folders rather than guessing a Kaggle dataset slug.
    from src.data_prep import ALIASES, normalize_label
    candidates = {}
    for search in search_roots:
        search = Path(search).resolve()
        if not search.is_dir():
            continue
        for folder, _, filenames in os.walk(search):
            if not any(name.lower().endswith('.wav') and not name.startswith('._') for name in filenames):
                continue
            current = Path(folder)
            matches = []
            while current.is_relative_to(search):
                key = normalize_label(current.name)
                if key in ALIASES:
                    matches.append((current, ALIASES[key]))
                if current == search:
                    break
                current = current.parent
            if matches:
                # Highest recognized class folder preserves nested parent/subclass layouts.
                class_dir, parent_label = matches[-1]
                candidates.setdefault(class_dir.parent, set()).add(parent_label)
    return sorted(path for path, labels in candidates.items() if len(labels) == 22)


if DOWNLOAD_DATASEC:
    if DATA_ROOT is not None:
        raise ValueError('Choose an existing DATA_ROOT or DOWNLOAD_DATASEC=True, not both.')
    downloaded = download_and_extract(SETUP_DIR)
    search_roots = [downloaded]
else:
    search_roots = [Path('/kaggle/input'), SETUP_DIR / 'extracted', Path.cwd() / 'data' / 'DataSEC']

if DATA_ROOT is None:
    candidates = find_datasec_roots(search_roots)
    if len(candidates) == 1:
        DATA_ROOT = candidates[0]
    elif len(candidates) > 1:
        raise ValueError('Multiple DataSEC copies found. Set DATA_ROOT to one of:\n' + '\n'.join(map(str, candidates)))
    else:
        raise FileNotFoundError('DataSEC audio is not attached or its 22-class layout was not recognized. '
                                'If you have no dataset, set DOWNLOAD_DATASEC=True above and rerun this cell. '
                                'If it is already attached, set DATA_ROOT to its extracted audio folder. '
                                'The GitHub repository contains code, not the audio dataset.')
DATA_ROOT = Path(DATA_ROOT).expanduser().resolve()
if not DATA_ROOT.is_dir() or not any(p.suffix.lower() == '.wav' for p in DATA_ROOT.rglob('*') if p.is_file()):
    raise FileNotFoundError(f'No extracted WAV files found in {DATA_ROOT}. Select the audio folder, not a ZIP or code folder.')
print('DATA_ROOT =', DATA_ROOT)


### Shared split paths
Reuse the team's agreed cleaned inventory, manifest and label mapping. The SincNet export is supported. Paths in the CSV must resolve under the `DATA_ROOT` printed above; if necessary, set `DATA_ROOT` explicitly to the root level used by that manifest. Leave both shared paths as `None` only when creating a candidate split for team agreement.


In [ ]:
# Optional in Google Colab when files are in Google Drive:
# from google.colab import drive
# drive.mount("/content/drive")

SHARED_SPLIT = None      # e.g. Path("/kaggle/input/team-splits/split_manifest.csv")
SHARED_LABEL_MAP = None  # e.g. Path("/kaggle/input/team-splits/class_mapping.json")
PREPARED = Path("outputs/prepared")
RUN = Path("outputs/efficientnet_b0")

assert (SHARED_SPLIT is None) == (SHARED_LABEL_MAP is None), "Set both shared paths or neither."

def run_module(module, *args):
    subprocess.run([sys.executable, "-m", module, *map(str, args)], check=True)


## 3. Audit the audio and preserve/create the split
Preparation stops for ambiguous labels, missing/corrupt audio or leakage. If a **new** inventory contains exact duplicates, agree exclusions with teammates before setting `DROP_EXACT_DUPLICATES=True`. Importing a shared split never drops recordings or changes assignments. Existing output folders are preserved; use a new `PREPARED` folder for a new preparation.


In [ ]:
DROP_EXACT_DUPLICATES = False
args = ["--data_dir", DATA_ROOT, "--out_dir", PREPARED]
if SHARED_SPLIT is not None:
    assert not DROP_EXACT_DUPLICATES, "Shared manifests must retain their agreed inventory."
    args += ["--shared_split", SHARED_SPLIT, "--label_map", SHARED_LABEL_MAP]
elif DROP_EXACT_DUPLICATES:
    args += ["--drop_duplicates"]
run_module("src.data_prep", *args)


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

manifest = pd.read_csv(PREPARED / "splits.csv")
display(pd.read_csv(PREPARED / "class_distribution.csv"))
print(json.dumps(json.loads((PREPARED / "audit.json").read_text()), indent=2))
pd.crosstab(manifest.label, manifest.split).reindex(columns=["train", "val", "test"]).plot(
    kind="bar", stacked=True, figsize=(13, 4))
plt.ylabel("Recordings")
plt.title("Agreed recording-level split: class distribution")
plt.tight_layout()
plt.show()


## 4. Inspect a deterministic log-Mel input
Mono audio is resampled to 32 kHz and cropped/padded to 5 seconds. FFT 1,024, hop 320 and 128 Mel bins yield `1 × 128 × 501`; the model resizes this to `224 × 224` and repeats it across three channels. A training-only mean/std is calculated during training. This preview shows an unnormalized centre crop.


In [ ]:
from src.dataset import AudioConfig, LogMelDataset
from IPython.display import Audio, display
sample = manifest[manifest.split == "train"].iloc[[0]]
preview = LogMelDataset(sample, DATA_ROOT, AudioConfig(), train=False)
x, label_id = preview[0]
print("Shape:", tuple(x.shape), "Class:", sample.iloc[0].label)
plt.figure(figsize=(10, 3))
plt.imshow(x[0].numpy(), origin="lower", aspect="auto", cmap="magma")
plt.xlabel("Time frame")
plt.ylabel("Mel bin")
plt.colorbar(label="dB")
plt.tight_layout()
plt.show()
display(Audio(filename=str(DATA_ROOT / sample.iloc[0].path)))


## 5. Run a short check
This checks loading, forward/backward passes, checkpoint saving and validation. It uses random initialization, only 64 recordings per split and two epochs. **These are not reportable project results.** Give every experiment a new output folder.


In [ ]:
common = ["--data_dir", DATA_ROOT, "--splits", PREPARED / "splits.csv",
          "--label_map", PREPARED / "label_map.json"]
run_module("src.train_efficientnet", *common, "--no_pretrained", "--limit", 64,
           "--epochs", 2, "--batch_size", 8, "--workers", 0, "--out_dir", "outputs/debug")


## 6. Train the full ImageNet-pretrained baseline
The classifier is Dropout(0.2) → Linear(1,280, 22). Training uses AdamW with separate head/backbone learning rates, cosine scheduling, training-only shift/gain/masking and mixup, and early stopping on validation macro-F1. The best validation checkpoint is restored. Test evaluation is a separate step.

For CUDA memory errors, reduce `BATCH_SIZE` and use a new run folder. GPU mixed precision is automatic. Record any changed settings in your report. Do not disconnect the runtime during training; this version saves the best model but does not implement optimizer-state resume.


In [ ]:
BATCH_SIZE = 32
EPOCHS = 30
run_module("src.train_efficientnet", *common, "--epochs", EPOCHS,
           "--batch_size", BATCH_SIZE, "--workers", 0, "--out_dir", RUN)


In [ ]:
from IPython.display import Image
history = pd.read_csv(RUN / "history.csv")
display(history.tail())
display(Image(filename=str(RUN / "learning_curves.png")))
print(json.dumps(json.loads((RUN / "metrics.json").read_text())["val"], indent=2))


## 7. Optional validation ablations
Uncomment the experiments you have time to run. Compare validation macro-F1 on the **same split**, then set `SELECTED_RUN` to the chosen experiment. Do not choose settings using test results.


In [ ]:
# run_module("src.train_efficientnet", *common, "--no_pretrained",
#            "--out_dir", "outputs/effnet_scratch", "--batch_size", BATCH_SIZE)
# run_module("src.train_efficientnet", *common, "--class_weights",
#            "--out_dir", "outputs/effnet_class_weights", "--batch_size", BATCH_SIZE)
SELECTED_RUN = RUN


## 8. Final held-out evaluation
Run this after fixing the experiment. Each run folder accepts one test export. The evaluator checks that the audio and manifest match training and refuses debug checkpoints. All four macro metrics use the full class list.


In [ ]:
run_module("src.evaluate_efficientnet", "--data_dir", DATA_ROOT,
           "--checkpoint", SELECTED_RUN / "best.pt", "--batch_size", BATCH_SIZE)
display(pd.read_csv(SELECTED_RUN / "comparison_row.csv"))
display(Image(filename=str(SELECTED_RUN / "test/confusion_matrix_normalized.png")))


## 9. Inspect class-wise performance and mistakes
Discuss low-recall classes, class imbalance, similar acoustic patterns and whether a five-second centre crop misses the event. Compare crop coverage and pretraining sources with teammates before attributing score differences to architecture alone. Scores are fractions from 0 to 1.


In [ ]:
per_class = pd.read_csv(SELECTED_RUN / "test/per_class_f1.csv")
display(per_class.sort_values("f1"))
errors = pd.read_csv(SELECTED_RUN / "test/misclassified.csv")
display(errors.head(10))
if len(errors):
    example = errors.iloc[0]
    print("True:", example.label, "Predicted:", example.pred)
    display(Audio(filename=str(DATA_ROOT / example.path)))


## 10. Predict another WAV and export your evidence
Prediction reloads the checkpoint's exact preprocessing and class map. A softmax score is not calibrated certainty. Download the resulting ZIP before a temporary Colab runtime ends. It contains only Model 2 run outputs, not the audio dataset or other models.


In [ ]:
NEW_AUDIO = None  # e.g. Path("/content/example.wav")
if NEW_AUDIO is not None:
    run_module("src.predict", "--checkpoint", SELECTED_RUN / "best.pt", "--audio", NEW_AUDIO)

import shutil
archive = shutil.make_archive(str(SELECTED_RUN) + "_results", "zip", root_dir=SELECTED_RUN)
print("Saved:", archive)
# In Google Colab, uncomment to download:
# from google.colab import files
# files.download(archive)


## Your report and presentation
Use the actual `config.json`, `metrics.json`, `comparison_row.csv`, learning curves and test figures. Describe the architecture, preprocessing, augmentation, validation selection, selected epoch, class errors, parameter count and measured timings. Report the shared split fingerprint and any differences in crop policy or hardware. The included CPU synthetic checks are software verification, not DataSEC experimental evidence.

Explain MBConv and squeeze-and-excitation, why a spectrogram can be passed to an image CNN, why macro-F1 matters for imbalanced classes, and why only training recordings determine normalization. Review all generated work and accurately declare LLM assistance.

References: [EfficientNet (Tan & Le, 2019)](https://proceedings.mlr.press/v97/tan19a.html), [torchvision model](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.efficientnet_b0.html), [DataSEC](https://zenodo.org/records/17033970), [SpecAugment](https://arxiv.org/abs/1904.08779), [mixup](https://arxiv.org/abs/1710.09412).
